In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    # f.read()가 파일 내용을 담은 문자열 객체를 새로 만들고, raw_text라는 변수가 그 객체를 가리킨다
    raw_text = f.read()

print("총 문자 개수: ", len(raw_text))
print(raw_text[:99])

총 문자 개수:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [4]:
import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)

# 'Hello' → ③ 'Hello'.strip()='Hello' → 내용 있음 → True  → ① 'Hello'.strip() → 'Hello' 담김
# ''      → ③ ''.strip()=''           → 빈값     → False → 버려짐 (①까지 안 감)
# ' '     → ③ ' '.strip()=''          → 공백벗기니 빈값 → False → 버려짐
# 'world' → ③ 'world'.strip()='world' → 내용 있음 → True  → ① 담김
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed)) # 4690

print(preprocessed[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [5]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [ ]:
# 코드 2-2 어휘사전 만들기

vocab = {token:integer for integer, token in enumerate(all_words)}

for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [9]:
# 코드 2-3 간단한 텍스트 토크나이저 구현

class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        # 1) 정수 ID 리스트를 다시 토큰(문자열)으로 바꾸고, 공백 한 칸으로 이어 붙인다.
        #    예: [12, 5, 40, 3] -> ["Hello", ",", "world", "."] -> "Hello , world ."
        #    -> 구두점 앞에도 공백이 들어가 버려서 어색하다.
        text = " ".join([self.int_to_str[i] for i in ids])

        # 2) re.sub(패턴, 치환문자열, 대상) : 패턴에 맞는 부분을 치환문자열로 바꾼다.
        #
        #    패턴  r'\s+([,.?!"()\'])'
        #      \s+          : 공백(스페이스, 탭 등)이 1개 이상 연속된 부분
        #      (...)        : 괄호 = "그룹". 매칭된 내용을 나중에 \1 로 다시 꺼내 쓸 수 있다.
        #      [,.?!"()\']  : 대괄호 = "이 중 한 글자". 즉 , . ? ! " ( ) ' 중 하나
        #      => 의미: "공백 다음에 구두점이 오는 자리"를 찾아라
        #
        #    치환문자열  r'\1'
        #      \1 : 첫 번째 그룹, 즉 위에서 잡힌 구두점 한 글자
        #      => 의미: (공백 + 구두점) 전체를 (구두점만) 으로 바꿔라 = 앞의 공백을 없앤다
        #
        #    r'' (raw string) 는 \s, \1 같은 역슬래시를 파이썬이 이스케이프로
        #    해석하지 않고 정규식 엔진에 그대로 넘기기 위한 표기.
        #
        #    예: "Hello , world ." -> "Hello, world."
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [14]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)
print(ids)

print(tokenizer.decode(ids))

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [15]:
text = "Hello, do you like tea?"
print(tokenizer.encode(text))

KeyError: 'Hello'